<a href="https://www.kaggle.com/code/page0526/tcga-blca-survival-analysis?scriptVersionId=326771773" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install lifelines timm==0.9.16

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 32.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 69.0 MB/s eta 0:00:00:00:010:01
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=1b45ab847bc6e77c43d4f7b514bfbc81ad4caa78c99f30d98768fbc08b8b1716
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: timm
    Found existing installation: timm 1.0.26
    Uninstalling timm-1.0.26:
      Successfully uninstalle

# TCGA-BRCA-survival analysis EDA

In [2]:
import pandas as pd
labels = pd.read_csv("/kaggle/input/datasets/jmalagontorres/tcga-brca-survival-analysis/clinical_data(labels).csv")
labels

,bcr_patient_barcode,Time,age_at_initial_pathologic_diagnosis,lymph_node_examined_count,vital_status,tissue_prospective_collection_indicator_YES,radiation_therapy_NO,breast_carcinoma_surgical_procedure_name_Lumpectomy,breast_carcinoma_surgical_procedure_name_Other,breast_carcinoma_surgical_procedure_name_Simple Mastectomy,...,pathologic_N_N1,pathologic_N_N2,pathologic_N_N3,pathologic_N_NX,pathologic_M_M1,pathologic_M_MX,pathologic_stage_Stage I,pathologic_stage_Stage III,pathologic_stage_Stage IV,pathologic_stage_Stage X
0,TCGA-E2-A1BD,1133.0,53,1.0,1,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
1,TCGA-BH-A0AW,622.0,56,12.0,1,False,False,True,False,False,...,True,False,False,False,False,False,False,False,False,False
2,TCGA-AO-A0JB,1542.0,50,14.0,1,False,False,False,False,False,...,True,False,False,False,False,False,False,True,False,False
3,TCGA-D8-A1JN,620.0,80,13.0,1,True,True,False,False,False,...,False,False,True,False,False,True,False,True,False,False
4,TCGA-EW-A1P8,239.0,58,15.0,2,False,True,True,False,False,...,False,False,True,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1058,TCGA-D8-A1XW,1309.0,53,1.0,1,True,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1059,TCGA-A8-A09M,1006.0,75,9.0,1,False,False,False,True,False,...,False,False,True,False,False,False,False,True,False,False
1060,TCGA-D8-A27W,373.0,55,12.0,1,True,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1061,TCGA-LL-A440,759.0,61,4.0,1,True,True,False,False,True,...,False,False,False,False,False,True,True,False,False,False


In [3]:
labels['vital_status'].value_counts()

vital_status
1    914
2    149
Name: count, dtype: int64

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")

In [5]:
import os
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import transforms

In [6]:
class PatchDataset(Dataset):
    def __init__(self, patch_paths, transform):
        self.patch_paths = patch_paths
        self.transform = transform

    def __len__(self):
        return len(self.patch_paths)

    def __getitem__(self, idx):
        img = Image.open(self.patch_paths[idx]).convert("RGB")
        img = self.transform(img)
        return img

In [7]:
root_dir = "/kaggle/input/datasets/jmalagontorres/tcga-brca-survival-analysis/WSIs"
clinical_csv = "/kaggle/input/datasets/jmalagontorres/tcga-brca-survival-analysis/clinical_data(labels).csv"
transforms = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
SAVE_DIR = "/kaggle/working/uni_features"
os.makedirs(SAVE_DIR, exist_ok=True)

In [8]:
import timm
print(timm.__version__)
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from huggingface_hub import login

login(hf_token)

device = "cuda" if torch.cuda.is_available() else "cpu"
# pretrained=True needed to load UNI2-h weights (and download weights for the first time)
timm_kwargs = {
            'img_size': 224, 
            'patch_size': 14, 
            'depth': 24,
            'num_heads': 24,
            'init_values': 1e-5, 
            'embed_dim': 1536,
            'mlp_ratio': 2.66667*2,
            'num_classes': 0, 
            'no_embed_class': True,
            'mlp_layer': timm.layers.SwiGLUPacked, 
            'act_layer': torch.nn.SiLU, 
            'reg_tokens': 8, 
            'dynamic_img_size': True
        }
encoder = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, **timm_kwargs).to(device)
transform = create_transform(**resolve_data_config(encoder.pretrained_cfg, model=encoder))
encoder.eval()

0.9.16


config.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.73G [00:00<?, ?B/s]

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1536, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((1536,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=1536, out_features=4608, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1536, out_features=1536, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((1536,), eps=1e-06, elementwise_affine=True)
      (mlp): GluMlp(
        (fc1): Linear(in_features=1536, out_features=8192, bias=True)
        (act): SiLU()
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): Identity()
    

In [9]:
from torch.utils.data import DataLoader
from tqdm import tqdm

clinical = pd.read_csv(clinical_csv)

clinical = clinical.set_index(
    "bcr_patient_barcode"
)

EXISTING_DIR = (
    "/kaggle/input/datasets/page0526/"
    "tcga-blca-embed-features/uni_features"
)

existing_patients = set()

if os.path.exists(EXISTING_DIR):

    existing_patients = set([
        f.replace(".pt", "")
        for f in os.listdir(EXISTING_DIR)
        if f.endswith(".pt")
    ])

print(
    f"Found {len(existing_patients)} "
    f"existing embeddings."
)


patient_ids = sorted([
    d for d in os.listdir(root_dir)
    if os.path.isdir(
        os.path.join(root_dir, d)
    )
])

for patient_id in tqdm(patient_ids):

    if patient_id in existing_patients:

        print(
            f"Skipping {patient_id}"
        )

        continue

    if patient_id not in clinical.index:

        print(
            f"No label found for "
            f"{patient_id}"
        )

        continue

    row = clinical.loc[patient_id]

    time = float(row["Time"])

    event = int(
        row["vital_status"] == 2
    )

    patient_dir = os.path.join(
        root_dir,
        patient_id
    )

    patch_paths = sorted([
        os.path.join(patient_dir, f)
        for f in os.listdir(patient_dir)
        if f.endswith(".jpg")
    ])

    if len(patch_paths) == 0:

        print(
            f"No patches for "
            f"{patient_id}"
        )

        continue

    dataset = PatchDataset(
        patch_paths,
        transform
    )

    loader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    features = []

    with torch.no_grad():

        for images in loader:

            images = images.to(device)

            feats = encoder(images)

            features.append(
                feats.cpu()
            )

    features = torch.cat(
        features,
        dim=0
    )

    # optional storage reduction
    features = features.half()

    print(
        patient_id,
        features.shape
    )

    torch.save(
        {
            "features": features,
            "patch_paths": patch_paths,
            "time": time,
            "event": event
        },
        os.path.join(
            SAVE_DIR,
            f"{patient_id}.pt"
        )
    )

Found 394 existing embeddings.


  0%|          | 0/1021 [00:00<?, ?it/s]

Skipping TCGA-3C-AALI
Skipping TCGA-3C-AALJ
Skipping TCGA-3C-AALK
Skipping TCGA-4H-AAAK
Skipping TCGA-5L-AAT1
Skipping TCGA-5T-A9QA
Skipping TCGA-A1-A0SB
Skipping TCGA-A1-A0SD
Skipping TCGA-A1-A0SE
Skipping TCGA-A1-A0SF
Skipping TCGA-A1-A0SH
Skipping TCGA-A1-A0SI
Skipping TCGA-A1-A0SJ
Skipping TCGA-A1-A0SK
Skipping TCGA-A1-A0SM
Skipping TCGA-A1-A0SN
Skipping TCGA-A1-A0SP
Skipping TCGA-A1-A0SQ
Skipping TCGA-A2-A04N
Skipping TCGA-A2-A04P
Skipping TCGA-A2-A04Q
Skipping TCGA-A2-A04R
Skipping TCGA-A2-A04T
Skipping TCGA-A2-A04U
Skipping TCGA-A2-A04V
Skipping TCGA-A2-A04W
Skipping TCGA-A2-A04X
Skipping TCGA-A2-A04Y
Skipping TCGA-A2-A0CK
Skipping TCGA-A2-A0CL
Skipping TCGA-A2-A0CM
Skipping TCGA-A2-A0CO
Skipping TCGA-A2-A0CP
Skipping TCGA-A2-A0CQ
Skipping TCGA-A2-A0CR
Skipping TCGA-A2-A0CS
Skipping TCGA-A2-A0CT
Skipping TCGA-A2-A0CU
Skipping TCGA-A2-A0CV
Skipping TCGA-A2-A0CW
Skipping TCGA-A2-A0CX
Skipping TCGA-A2-A0CY
Skipping TCGA-A2-A0CZ
Skipping TCGA-A2-A0D0
Skipping TCGA-A2-A0D3
Skipping T

 39%|███▊      | 395/1021 [01:54<03:00,  3.46it/s]

TCGA-AR-A24Q torch.Size([1148, 1536])


...

TCGA-XX-A89A torch.Size([311, 1536])


100%|█████████▉| 1020/1021 [10:34:53<00:40, 40.61s/it]

TCGA-Z7-A8R5 torch.Size([350, 1536])


100%|██████████| 1021/1021 [10:36:16<00:00, 37.39s/it]

TCGA-Z7-A8R6 torch.Size([777, 1536])


In [10]:
!zip -r /kaggle/working/uni_features.zip /kaggle/working/uni_features

  adding: kaggle/working/uni_features/ (stored 0%)
  adding: kaggle/working/uni_features/TCGA-GM-A3NW.pt (deflated 11%)
  adding: kaggle/working/uni_features/TCGA-BH-A0HW.pt (deflated 11%)
  adding: kaggle/working/uni_features/TCGA-E2-A1B4.pt (deflated 11%)
  adding: kaggle/working/uni_features/TCGA-D8-A1XB.pt (deflated 11%)
  adding: kaggle/working/uni_features/TCGA-AR-A24R.pt (deflated 11%)
  adding: kaggle/working/uni_features/TCGA-EW-A3E8.pt (deflated 11%)
  adding: kaggle/working/uni_features/TCGA-B6-A0IK.pt (deflated 11%)
  adding: kaggle/working/uni_features/TCGA-EW-A1J1.pt (deflated 11%)
  adding: kaggle/working/uni_features/TCGA-D8-A1JA.pt (deflated 11%)
  adding: kaggle/working/uni_features/TCGA-BH-A0E2.pt (deflated 11%)
  adding: kaggle/working/uni_features/TCGA-B6-A0WT.pt (deflated 11%)
  adding: kaggle/working/uni_features/TCGA-HN-A2NL.pt (deflated 11%)
  adding: kaggle/working/uni_features/TCGA-LL-A5YN.pt (deflated 11%)
  adding: kaggle/working/uni_features/TCGA-BH-A2L8.p

In [11]:
from sklearn.model_selection import train_test_split
import os

feature_dir = "/kaggle/working/uni_features"

all_files = sorted([
    os.path.join(feature_dir, f)
    for f in os.listdir(feature_dir)
    if f.endswith(".pt")
])

# 70% train, 30% temp
train_files, temp_files = train_test_split(
    all_files,
    test_size=0.30,
    random_state=42
)

# split temp into 15% val, 15% test
val_files, test_files = train_test_split(
    temp_files,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_files))
print("Val:", len(val_files))
print("Test:", len(test_files))

Train: 424
Val: 91
Test: 91


In [12]:
class BLCADataset(Dataset):

    def __init__(self, files):
        self.files = files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        sample = torch.load(
            self.files[idx],
            map_location="cpu"
        )

        return {
            "features": sample["features"].float(),
            "time": torch.tensor(sample["time"]).float(),
            "event": torch.tensor(sample["event"]).float()
        }

In [31]:
trainset = BLCADataset(train_files)
valset = BLCADataset(val_files)
testset = BLCADataset(test_files)

In [32]:
print(len(trainset), len(valset), len(testset))

424 91 91


In [33]:
def collate_fn(batch):

    features = [x["features"] for x in batch]

    times = torch.stack([
        x["time"] for x in batch
    ])

    events = torch.stack([
        x["event"] for x in batch
    ])

    return features, times, events

In [16]:
class GatedABMIL(nn.Module):

    def __init__(
        self,
        input_dim=1536,
        hidden_dim=256
    ):
        super().__init__()

        self.attn_V = nn.Linear(
            input_dim,
            hidden_dim
        )

        self.attn_U = nn.Linear(
            input_dim,
            hidden_dim
        )

        self.attn_w = nn.Linear(
            hidden_dim,
            1
        )

        self.risk_head = nn.Linear(
            input_dim,
            1
        )

    def forward(self, x):

        A_V = torch.tanh(
            self.attn_V(x)
        )

        A_U = torch.sigmoid(
            self.attn_U(x)
        )

        A = self.attn_w(
            A_V * A_U
        )

        A = torch.softmax(
            A.squeeze(1),
            dim=0
        )

        M = torch.sum(
            x * A.unsqueeze(1),
            dim=0
        )

        risk = self.risk_head(M)

        return risk.squeeze(), A

In [17]:
def cox_loss(risk, time, event):

    order = torch.argsort(
        time,
        descending=True
    )

    risk = risk[order]
    event = event[order]

    log_risk = torch.logcumsumexp(
        risk,
        dim=0
    )

    loss = -(risk - log_risk)

    loss = loss * event

    return loss.sum() / (
        event.sum() + 1e-8
    )

In [37]:
trainloader = DataLoader(
    trainset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

valloader = DataLoader(
    valset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)

testloader = DataLoader(
    testset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)

In [38]:
print(len(trainloader), len(valloader), len(testloader))
next(iter(trainloader))

106 23 23


([tensor([[ 1.3477, -0.1693, -0.6372,  ..., -0.0371,  0.0337, -0.3501],
          [ 1.2930,  0.2783, -0.4575,  ..., -0.0940, -0.8096, -0.4983],
          [ 0.7998,  0.1855, -1.1172,  ...,  0.4417, -0.7148,  0.0638],
          ...,
          [ 0.9507, -0.0391, -0.5068,  ...,  0.2003, -0.5356, -0.2448],
          [ 1.0811, -0.1530, -0.3467,  ...,  0.3555, -0.5098, -0.4148],
          [ 0.4026, -0.2159, -0.5698,  ...,  0.4294, -0.9795, -0.1842]]),
  tensor([[ 1.3516, -0.7559, -0.5947,  ...,  0.3516, -0.8008, -0.0405],
          [ 0.7114, -0.4858, -0.4058,  ...,  0.5356,  0.4958,  0.2290],
          [ 1.0752, -0.3694, -0.7012,  ..., -0.1648, -0.8223, -0.0555],
          ...,
          [ 0.0881, -0.5166, -0.0138,  ..., -0.2151, -0.8252,  0.0536],
          [ 0.4275, -0.3677, -0.1364,  ...,  0.4487, -0.8428,  0.2057],
          [ 1.3604, -0.6621, -0.3132,  ...,  0.0469, -0.4050,  0.2050]]),
  tensor([[ 0.5425, -0.7739,  0.1222,  ...,  0.1655, -0.6606,  0.1824],
          [ 0.1901,  0.2776,  

In [39]:
model = GatedABMIL(
    input_dim=1536
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

In [40]:
EPOCHS = 50
for epoch in range(50):

    model.train()

    epoch_loss = 0

    for bags, times, events in trainloader:

        times = times.to(device)
        events = events.to(device)

        risks = []

        for bag in bags:

            bag = bag.to(device)

            risk, _ = model(bag)

            risks.append(risk)

        risks = torch.stack(risks)

        loss = cox_loss(
            risks,
            times,
            events
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    print(
        f"Epoch {epoch}: "
        f"{epoch_loss:.4f}"
    )

Epoch 0: 28.2386
Epoch 1: 34.4607
Epoch 2: 32.1152
Epoch 3: 31.6591
Epoch 4: 31.4372
Epoch 5: 33.4273
Epoch 6: 27.1966
Epoch 7: 25.1172
Epoch 8: 21.4624
Epoch 9: 22.4226
Epoch 10: 18.0497
Epoch 11: 19.4720
Epoch 12: 18.7344
Epoch 13: 15.0096
Epoch 14: 17.1545
Epoch 15: 14.4515
Epoch 16: 15.9034
Epoch 17: 11.9436
Epoch 18: 11.9787
Epoch 19: 9.7480
Epoch 20: 6.1818
Epoch 21: 11.8931
Epoch 22: 12.3413
Epoch 23: 10.3964
Epoch 24: 9.7417
Epoch 25: 8.6857
Epoch 26: 9.8341
Epoch 27: 8.9411
Epoch 28: 8.3601
Epoch 29: 8.4507
Epoch 30: 7.0574
Epoch 31: 11.0658
Epoch 32: 5.0839
Epoch 33: 6.6891
Epoch 34: 8.1252
Epoch 35: 6.2921
Epoch 36: 3.4756
Epoch 37: 6.3822
Epoch 38: 4.9761
Epoch 39: 7.5398
Epoch 40: 9.5180
Epoch 41: 6.8561
Epoch 42: 5.4952
Epoch 43: 6.8432
Epoch 44: 5.5919
Epoch 45: 9.3499
Epoch 46: 4.9340
Epoch 47: 4.0550
Epoch 48: 5.7691
Epoch 49: 3.8123


In [41]:
all_risk = []
all_time = []
all_event = []

model.eval()

with torch.no_grad():

    for bags, times, events in valloader:

        for bag in bags:

            risk, _ = model(
                bag.cuda()
            )

            all_risk.append(
                risk.item()
            )

        all_time.extend(
            times.numpy()
        )

        all_event.extend(
            events.numpy()
        )

In [42]:
from lifelines.utils import concordance_index
cindex = concordance_index(
    all_time,
    [-x for x in all_risk],
    all_event
)

print(cindex)

0.5168776371308017


# Saliency map

In [2]:
import torch
sample = torch.load(
    "/kaggle/working/uni_features/TCGA-AR-A24Q.pt"
)

features = sample["features"].float().cuda()

model.eval()

with torch.no_grad():
    risk, attention = model(features)

attention = attention.cpu()

topk = torch.topk(attention, k=20)

for rank, idx in enumerate(topk.indices.tolist(), start=1):

    print(
        rank,
        attention[idx].item(),
        sample["patch_paths"][idx]
    )

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/uni_features/TCGA-AR-A24Q.pt'